# Morphology Cell Translator — training curves and reconstruction diagnostics

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.stats import pearsonr

RUN_ROOT = Path(os.environ.get('CELL_TRANSLATOR_RUN_DIR', str(Path.cwd() / 'outputs' / 'run_70_15_15_qc_v3'))).expanduser().resolve()
AE_DIR = RUN_ROOT / 'autoencoders'
DEV_DIR = RUN_ROOT / 'cyclegan_ipsc_astrocyte_drugs'
CTRL_DIR = RUN_ROOT / 'cyclegan_control_to_perturbation'
OUT = RUN_ROOT / 'evaluation' / 'training_and_reconstruction_plots'
OUT.mkdir(parents=True, exist_ok=True)
DRUGS = ['DMSO','Latrunculin_B','GW3965','Clozapine']
CONDITIONS = [f'{cell}_{drug}' for cell in ['iPSC','Astrocytes'] for drug in DRUGS]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,
                     'pdf.fonttype':42, 'ps.fonttype':42})
print('Root:', RUN_ROOT, '| Device:', DEVICE)


## 1. Autoencoder training and validation losses

In [ ]:
def savefig(fig, filename):
    fig.savefig(OUT / f'{filename}.png', dpi=300, bbox_inches='tight')
    fig.savefig(OUT / f'{filename}.pdf', bbox_inches='tight')
    plt.close(fig)

summary_path = AE_DIR/'ae_test_summary.csv'
ae_summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
fig, axes = plt.subplots(2, 4, figsize=(17, 7.5), sharex=True)
for ax, cond in zip(axes.flat, CONDITIONS):
    f = AE_DIR/f'{cond}_training_history.csv'
    if not f.exists():
        ax.text(.5,.5,'History not found',transform=ax.transAxes,ha='center'); ax.set_title(cond); continue
    h = pd.read_csv(f)
    ep = h['epoch'].to_numpy() if 'epoch' in h else np.arange(1,len(h)+1)
    ax.plot(ep,h['train_loss'],label='Train',lw=1.7)
    ax.plot(ep,h['validation_loss'],label='Validation',lw=1.7)
    if not ae_summary.empty and cond in set(ae_summary['condition']):
        best=int(ae_summary.loc[ae_summary.condition==cond,'best_epoch'].iloc[0])
        ax.axvline(best,color='0.35',lw=.9,ls='--')
    ax.set_title(cond.replace('_',' '),fontsize=10)
    ax.grid(alpha=.2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('AE objective')
handles,labels=axes.flat[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',ncol=2,bbox_to_anchor=(.5,1.03),frameon=False)
fig.suptitle('Autoencoder: training vs validation objective',y=1.07,fontsize=14)
fig.tight_layout(); savefig(fig,'AE_training_validation_8_conditions')
print('Saved AE training / validation curves')


## 2. Exact autoencoder reconstruction on held-out test cells


In [ ]:
class MorphologyAutoencoder(nn.Module):
    def __init__(self,input_dim,latent_dim=128):
        super().__init__()
        def enc_block(i,o):
            return nn.Sequential(nn.Linear(i,o),nn.BatchNorm1d(o),nn.LeakyReLU(.1),nn.Dropout(.15))
        def dec_block(i,o,drop=True):
            layers=[nn.Linear(i,o),nn.BatchNorm1d(o),nn.LeakyReLU(.1)]
            if drop: layers.append(nn.Dropout(.15))
            return nn.Sequential(*layers)
        self.encoder=nn.Sequential(enc_block(input_dim,1024),enc_block(1024,512),
            nn.Sequential(nn.Linear(512,256),nn.BatchNorm1d(256),nn.LeakyReLU(.1)),nn.Linear(256,latent_dim))
        self.decoder=nn.Sequential(dec_block(latent_dim,256),dec_block(256,512),
            dec_block(512,1024,drop=False),nn.Linear(1024,input_dim))
    def forward(self,x):
        z=self.encoder(x)
        return z,self.decoder(z)

def reconstruct_condition(cond,batch_size=128):
    prefix=AE_DIR/cond
    f_ckpt=Path(str(prefix)+'_ae_checkpoint.pt')
    f_x=Path(str(prefix)+'_X_test_scaled.npy')
    f_stats=Path(str(prefix)+'_scaler_stats.csv')
    ckpt=torch.load(f_ckpt,map_location=DEVICE,weights_only=False) # trusted local notebook checkpoint only
    X=np.load(f_x).astype(np.float32)
    stats=pd.read_csv(f_stats)
    assert X.ndim==2 and X.shape[1]==len(stats)
    cfg=ckpt['config']
    model=MorphologyAutoencoder(X.shape[1],int(cfg['latent_dim'])).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'],strict=True)
    model.eval()
    batches=[]
    with torch.no_grad():
        for i in range(0,len(X),batch_size):
            _,y=model(torch.as_tensor(X[i:i+batch_size],device=DEVICE))
            batches.append(y.cpu().numpy())
    R=np.concatenate(batches).astype(np.float64)
    X=X.astype(np.float64)
    assert np.isfinite(X).all() and np.isfinite(R).all()
    mu=stats.scaler_mean.to_numpy(dtype=float)
    sd=stats.scaler_scale.to_numpy(dtype=float)
    assert np.isfinite(sd).all() and (sd>0).all()
    return X*sd+mu,R*sd+mu,stats.feature.astype(str).tolist()

def safe_r(x,y):
    if len(x)<2 or np.std(x)<1e-12 or np.std(y)<1e-12: return np.nan
    return float(np.corrcoef(x,y)[0,1])

rows=[]
fig,axes=plt.subplots(2,4,figsize=(17,7.5),sharex=True)
for ax,cond in zip(axes.flat,CONDITIONS):
    try:
        X,R,features=reconstruct_condition(cond)
    except FileNotFoundError as exc:
        ax.text(.5,.5,f'Missing: {exc.name}',transform=ax.transAxes,ha='center',fontsize=8);continue
    # R² for each morphology feature; no clipping.
    ss_res=np.square(X-R).sum(axis=0)
    ss_tot=np.square(X-X.mean(axis=0)).sum(axis=0)
    r2=np.full(X.shape[1],np.nan)
    valid=ss_tot>1e-12
    r2[valid]=1-ss_res[valid]/ss_tot[valid]
    v=r2[np.isfinite(r2)]
    mu_r=safe_r(X.mean(0),R.mean(0))
    sd_r=safe_r(X.std(0),R.std(0))
    rows.append(dict(condition=cond,n_test=len(X),n_features=X.shape[1],n_valid_r2=len(v),
                     median_feature_r2=np.median(v) if len(v) else np.nan,
                     fraction_features_r2_gt_05=np.mean(v>.5) if len(v) else np.nan,
                     feature_mean_pearson_original=mu_r,
                     feature_sd_pearson_original=sd_r,
                     overall_value_pearson_original=safe_r(X.ravel(),R.ravel())))
    ax.hist(v,bins=np.linspace(max(-2,np.nanpercentile(v,1)),1,45),color='#4A82A8',alpha=.85)
    ax.axvline(.5,color='#C46A52',ls='--',lw=1)
    ax.set_title(cond.replace('_',' '),fontsize=10)
    ax.set_xlabel('Per-feature test R²')
    ax.set_ylabel('Feature count')
    ax.text(.04,.96,f'Median R² = {np.median(v):.3f}\nMean r = {mu_r:.3f}\nSD r = {sd_r:.3f}',
            transform=ax.transAxes,ha='left',va='top',fontsize=8,
            bbox=dict(boxstyle='round',facecolor='white',alpha=.85,edgecolor='0.8'))
fig.suptitle('Autoencoder held-out reconstruction — feature R²',fontsize=14)
fig.tight_layout();savefig(fig,'AE_test_feature_R2_distributions')
recon_summary=pd.DataFrame(rows)
recon_summary.to_csv(OUT/'AE_reconstruction_test_summary_original_scale.csv',index=False)
display(recon_summary.round(3))


### Reconstruction mean and SD scatterplots (optional)

In [ ]:
fig,axs=plt.subplots(2,4,figsize=(17,8))
for ax,cond in zip(axs.flat,CONDITIONS):
    try:X,R,_=reconstruct_condition(cond)
    except FileNotFoundError:continue
    xm,ym=X.mean(0),R.mean(0)
    ax.scatter(xm,ym,s=3,alpha=.25,rasterized=True)
    lo=min(xm.min(),ym.min());hi=max(xm.max(),ym.max())
    ax.plot([lo,hi],[lo,hi],ls='--',color='0.4',lw=1)
    ax.set_title(f'{cond} | r={safe_r(xm,ym):.3f}',fontsize=9)
    ax.set_xlabel('Measured feature mean');ax.set_ylabel('Reconstructed mean')
fig.tight_layout();savefig(fig,'AE_test_feature_means_original_scale')
fig,axs=plt.subplots(2,4,figsize=(17,8))
for ax,cond in zip(axs.flat,CONDITIONS):
    try:X,R,_=reconstruct_condition(cond)
    except FileNotFoundError:continue
    xs,ys=X.std(0),R.std(0)
    ax.scatter(xs,ys,s=3,alpha=.25,rasterized=True)
    hi=max(xs.max(),ys.max())
    ax.plot([0,hi],[0,hi],ls='--',color='0.4',lw=1)
    ax.set_title(f'{cond} | r={safe_r(xs,ys):.3f}',fontsize=9)
    ax.set_xlabel('Measured feature SD');ax.set_ylabel('Reconstructed SD')
fig.tight_layout();savefig(fig,'AE_test_feature_SDs_original_scale')


## 3. CycleGAN losses — all 10 translations

In [ ]:
DEV_METRICS={'generator':'gen_SA','cycle':'cycle','w_gap':'w_gap',
             'identity':'identity','mmd':'mmd','cov':'cov','moment':'mom'}
CTRL_METRICS={'generator':'gen_control_to_drug','cycle':'cycle','w_gap':'wasserstein_gap',
              'identity':'identity','mmd':'mmd','cov':'cov','moment':'moment'}

def get_history(kind,name):
    if kind=='developmental':
        path=DEV_DIR/name/'training_history.csv';mapping=DEV_METRICS
    else:
        path=CTRL_DIR/name/'training_history.csv';mapping=CTRL_METRICS
    h=pd.read_csv(path)
    required=('generator','cycle','w_gap')
    missing=[mapping[k] for k in required if mapping[k] not in h.columns]
    if missing:raise ValueError(f'{path} missing {missing}. Available: {list(h.columns)}')
    return h,mapping

def plot_translation_grid(kind,names,filename):
    fig,axes=plt.subplots(3,len(names),figsize=(4.1*len(names),9),squeeze=False)
    for j,name in enumerate(names):
        fpath=(DEV_DIR/name if kind=='developmental' else CTRL_DIR/name)/'training_history.csv'
        if not fpath.exists():
            for ax in axes[:,j]:ax.text(.5,.5,'Missing history',transform=ax.transAxes,ha='center')
            continue
        h,m=get_history(kind,name)
        ep=h['epoch'].to_numpy() if 'epoch' in h else np.arange(1,len(h)+1)
        for i,(metric,ylab) in enumerate([('generator','Forward generator objective'),('cycle','Cycle-consistency loss'),('w_gap','Wasserstein critic gap')]):
            ax=axes[i,j]
            ax.plot(ep,h[m[metric]].to_numpy(),lw=1.4,color=['#345F89','#9B567E','#438C7C'][i])
            ax.grid(alpha=.2);ax.set_xlabel('Epoch')
            if j==0:ax.set_ylabel(ylab)
            if i==0:ax.set_title(name.replace('_',' '),fontsize=10)
    fig.suptitle(f'{kind.replace("_"," ").title()} CycleGAN — training histories',fontsize=14,y=1.015)
    fig.tight_layout();savefig(fig,filename)

plot_translation_grid('developmental',DRUGS,'CycleGAN_developmental_4_training_curves')
for cell in ['iPSC','Astrocytes']:
    names=[f'{cell}_DMSO_to_{d}' for d in DRUGS[1:]]
    plot_translation_grid('control_to_drug',names,f'CycleGAN_{cell}_control_to_drug_training_curves')
print('Saved 4 developmental + 6 control-to-drug loss curves')


## 4. Individual auxiliary CycleGAN loss components

In [ ]:
for kind,names,tag in [
    ('developmental',DRUGS,'developmental'),
    ('control_to_drug',[f'{c}_DMSO_to_{d}' for c in ['iPSC','Astrocytes'] for d in DRUGS[1:]],'control_to_drug')]:
    fig,axes=plt.subplots(4,1,figsize=(11,11),sharex=True)
    for name in names:
        f=(DEV_DIR/name if kind=='developmental' else CTRL_DIR/name)/'training_history.csv'
        if not f.exists():continue
        h,m=get_history(kind,name)
        ep=h['epoch'].to_numpy() if 'epoch' in h else np.arange(1,len(h)+1)
        for ax,key in zip(axes,['identity','mmd','cov','moment']):
            col=m[key]
            if col in h.columns:ax.plot(ep,h[col],lw=1,label=name.replace('_',' '))
    for ax,label in zip(axes,['Identity','MMD','Covariance','Moment']):
        ax.set_ylabel(label);ax.grid(alpha=.2)
    axes[0].legend(loc='upper right',fontsize=7,ncol=2,frameon=False)
    axes[-1].set_xlabel('Epoch')
    fig.suptitle(f'{tag}: auxiliary CycleGAN training terms')
    fig.tight_layout();savefig(fig,f'CycleGAN_{tag}_auxiliary_losses')
